<a href="https://www.kaggle.com/code/kaiyoo88/rocketlaunch-simulation-benchmark-flexibility?scriptVersionId=312974686" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🚀 Getting Started with Kaggle Benchmarks

Welcome! This notebook will teach you how to create, run, and evaluate LLM benchmarks using the `kaggle-benchmarks` library.

**Key concepts** 
1. Task: A Python function defining the problem (e.g., "Solve this riddle").
2. Run: The execution of a task
3. Benchmark: A collection of tasks that is arbitrarily put together by a user. There is no code implementation for this. This is a feature that Kaggle supports on the graphical user interface so that users can put together their own benchmarks based on the tasks that they care about

Now, let's dive into creating a task and executing your first run!


In [ ]:
# %choose batch_riddle_solver

# Static RocketLaunch Simulation - Task: Flexibility_stage_transition_weighted

In [5]:
import json
import random
from dataclasses import dataclass
from collections import defaultdict

import pandas as pd
import kaggle_benchmarks as kbench


# =========================================================
# CONFIG
# =========================================================
VERBOSE = True
TASK_FILE = "/kaggle/input/datasets/kaiyoo88/task-json/TASK_JSON.json"

# Transition steps receive larger weight
BASE_STEP_WEIGHT = 1.0
TRANSITION_STEP_WEIGHT = 2.5

# If True, the model returns action IDs like A01.
# If False, the model returns raw action names.
USE_ACTION_IDS = True

# Overall flexibility score weights
W_FLEX = 0.35
W_TRANSITION = 0.45
W_PERSEV = 0.20

# Fixed seed for reproducible shuffled action IDs
SEED = 2026
rng = random.Random(SEED)


# =========================================================
# LOAD TASKS
# =========================================================
with open(TASK_FILE, "r", encoding="utf-8") as f:
    TASKS = json.load(f)

steps = TASKS


# =========================================================
# SCHEMA
# =========================================================
@dataclass
class ActionOutput:
    action: str


# =========================================================
# ACTION CATALOG
# =========================================================
FUNCTION_TABLE = [
    ("launch_from_earth", "Start launch from Earth."),
    ("wait", "Wait without taking immediate action and keep the current state."),
    ("abort_mission", "Abort the entire mission."),
    ("continue_burn", "Continue the current propulsion burn."),
    ("reduce_throttle", "Reduce thrust to manage fuel use or attitude."),
    ("shutdown_engine", "Shut down the current engine."),
    ("separate_stage1", "Separate Stage 1."),
    ("coast_transfer", "Continue transfer/movement by coasting without thrust."),
    ("trajectory_correction_burn", "Perform a short burn for trajectory correction."),
    ("restart_stage2_engine", "Restart the Stage 2 engine."),
    ("abort_capture", "Abort the attempt to capture into the target orbit."),
    ("separate_stage2", "Separate Stage 2."),
    ("continue_orbit_hold", "Maintain the current orbit."),
    ("adjust_stage3_center_of_mass", "Adjust the center of mass for Stage 3 landing."),
    ("adjust_stage3_booster_usage", "Adjust booster usage for Stage 3 landing."),
    ("continue_descent", "Continue the current descent."),
    ("abort_landing", "Abort the current landing attempt."),
    ("continue_ice_breaking", "Continue the ice-breaking operation."),
    ("pause_ice_breaking", "Pause the ice-breaking operation."),
    ("retrieve_ice_robot", "Retrieve the ice-work robot."),
    ("insert_pipe", "Insert the pipe."),
    ("realign_pipe", "Realign the pipe."),
    ("start_water_lift", "Start lifting water."),
    ("remove_pipe", "Remove the inserted pipe."),
    ("start_electrolysis", "Start electrolysis."),
    ("abort_surface_process", "Abort the surface resource processing operation."),
    ("continue_fill_process", "Continue the fuel/sample filling process."),
    ("route_sample_to_return_module", "Route the sample toward the return module."),
    ("lock_s4_tube", "Lock the Stage 4 line."),
    ("lock_s5_tube", "Lock the Stage 5 line."),
    ("lock_sample_tube", "Lock the sample line."),
    ("lock_all_tubes", "Lock all major lines at once."),
    ("launch_return_stack", "Launch the return stack from the surface of Europa."),
    ("abort_return_launch", "Abort the launch departing from Europa."),
    ("adjust_stage4_booster_usage", "Adjust Stage 4 booster usage during Europa escape."),
    ("adjust_stage4_center_of_mass", "Adjust the Stage 4 center of mass during Europa escape."),
    ("separate_stage4", "Separate Stage 4."),
    ("separate_stage5", "Separate Stage 5."),
    ("continue_coast", "Continue the current unpowered coast."),
    ("ignite_return_module_booster", "Ignite the return module booster."),
    ("shutdown_return_module_booster", "Shut down the return module booster."),
    ("adjust_return_module_center_of_mass", "Adjust the center of mass for return module landing."),
    ("adjust_return_module_booster_usage", "Adjust booster usage for return module landing."),
]

# Shuffle once with a fixed seed so IDs do not leak mission order
shuffled_functions = FUNCTION_TABLE[:]
rng.shuffle(shuffled_functions)

ACTION_CATALOG = [
    {"id": f"A{i:02d}", "name": name, "desc": desc}
    for i, (name, desc) in enumerate(shuffled_functions, start=1)
]

NAME2ID = {a["name"]: a["id"] for a in ACTION_CATALOG}
ID2NAME = {a["id"]: a["name"] for a in ACTION_CATALOG}
ALL_IDS = [a["id"] for a in ACTION_CATALOG]


# =========================================================
# VALIDATE ACTION COVERAGE
# =========================================================
def validate_action_coverage(steps):
    catalog_names = set(NAME2ID.keys())

    missing_correct = sorted({
        step["correct_action"]
        for step in steps
        if step["correct_action"] not in catalog_names
    })

    missing_valid = sorted({
        action
        for step in steps
        for action in step["valid_actions"]
        if action not in catalog_names
    })

    print("=== ACTION COVERAGE CHECK ===")
    print("Missing correct_action:", missing_correct)
    print("Missing valid_actions :", missing_valid)

    if missing_correct or missing_valid:
        raise ValueError("ACTION_CATALOG is missing actions used in task_json.")


# =========================================================
# HELPERS
# =========================================================
def sanitize_state(state: dict) -> dict:
    """
    Remove leakage-prone metadata if present.
    """
    s = dict(state)
    s.pop("goal", None)
    s.pop("current_stage_index", None)
    return s


def get_stage_name(task_id: str) -> str:
    """
    Example:
      'stage1_01_launch_from_earth' -> 'stage1'
    """
    return task_id.split("_")[0]


def build_stage_action_map(steps):
    """
    Build a mapping:
      stage -> set of correct actions appearing in that stage
    """
    stage_to_actions = defaultdict(set)
    for step in steps:
        stage = get_stage_name(step["task_id"])
        stage_to_actions[stage].add(step["correct_action"])
    return dict(stage_to_actions)


def build_previous_stage_only_actions(steps, current_idx, stage_to_actions):
    """
    Return actions that appeared in previous stages but do not belong to the current stage.
    These are strong candidates for perseveration errors.
    """
    current_stage = get_stage_name(steps[current_idx]["task_id"])

    previous_stages = []
    seen = set()
    for j in range(current_idx):
        stage = get_stage_name(steps[j]["task_id"])
        if stage != current_stage and stage not in seen:
            previous_stages.append(stage)
            seen.add(stage)

    previous_actions = set()
    for stage in previous_stages:
        previous_actions |= stage_to_actions.get(stage, set())

    current_stage_actions = stage_to_actions.get(current_stage, set())
    previous_stage_only_actions = previous_actions - current_stage_actions
    return previous_stage_only_actions


def is_transition_step(steps, idx):
    """
    A transition step is the first step of a new stage.
    """
    if idx == 0:
        return False

    prev_stage = get_stage_name(steps[idx - 1]["task_id"])
    curr_stage = get_stage_name(steps[idx]["task_id"])
    return prev_stage != curr_stage


# =========================================================
# PROMPT
# =========================================================
def build_prompt(step):
    state = sanitize_state(step["state"])

    if USE_ACTION_IDS:
        action_block = json.dumps(
            [{"id": a["id"], "description": a["desc"]} for a in ACTION_CATALOG],
            ensure_ascii=False,
            indent=2,
        )
        output_spec = '{"action": "Axx"}'
    else:
        action_block = json.dumps(
            [{"name": a["name"], "description": a["desc"]} for a in ACTION_CATALOG],
            ensure_ascii=False,
            indent=2,
        )
        output_spec = '{"action": "action_name"}'

    return f"""
You are a mission planner for a Europa resource-return mission.

Choose exactly one best next action.

Important:
- The mission may enter a new stage with different control needs.
- When the stage changes, do not keep using policies that belonged to earlier stages.
- Avoid perseverating on earlier-stage behaviors when the current state calls for a different regime.

Rules:
- Return JSON only.
- Choose a single action from the action catalog.

Return JSON:
{output_spec}

CURRENT STATE:
{json.dumps(state, ensure_ascii=False, indent=2)}

ACTION CATALOG:
{action_block}
""".strip()


# =========================================================
# STEP-LEVEL FLEXIBILITY SCORING
# =========================================================
def score_flexibility_step(pred_action_name, step, previous_stage_only_actions, current_stage_actions):
    """
    Step-level flexibility score.

    Cases:
    1. Exact correct action -> 1.0
    2. Wrong but still belongs to current-stage action family -> 0.5
    3. Wrong and belongs to previous-stage-only actions -> 0.0
    4. Wrong but valid for this step -> 0.4
    5. Invalid / unrelated -> 0.2
    """
    correct = step["correct_action"]
    valid = set(step["valid_actions"])

    if pred_action_name == correct:
        return 1.0, "exact_correct"

    if pred_action_name in previous_stage_only_actions:
        return 0.0, "perseveration_error"

    if pred_action_name in current_stage_actions:
        return 0.5, "current_stage_but_wrong"

    if pred_action_name in valid:
        return 0.4, "valid_but_wrong"

    return 0.2, "invalid_or_unrelated"


# =========================================================
# CORE
# =========================================================
def run_flexibility_episode(llm, steps):
    total = len(steps)
    stage_to_actions = build_stage_action_map(steps)

    weighted_sum = 0.0
    weight_total = 0.0

    transition_sum = 0.0
    transition_count = 0

    perseveration_errors = 0
    wrong_total = 0

    logs = []

    with kbench.chats.new("europa-flexibility-stage-transition-chat"):
        for i, step in enumerate(steps):
            prompt = build_prompt(step)

            try:
                resp = llm.prompt(prompt, schema=ActionOutput)
                raw_action = resp.action if resp is not None and hasattr(resp, "action") else None
            except Exception as e:
                raw_action = None
                if VERBOSE:
                    print(f"[ERROR][{i+1:02d}/{total:02d}] prompt/parsing error: {e}")

            # Normalize to action name
            pred_action_name = None
            if raw_action in ID2NAME:
                pred_action_name = ID2NAME[raw_action]
            elif raw_action in NAME2ID:
                pred_action_name = raw_action

            current_stage = get_stage_name(step["task_id"])
            current_stage_actions = stage_to_actions.get(current_stage, set())
            previous_stage_only_actions = build_previous_stage_only_actions(steps, i, stage_to_actions)

            flex_score, label = score_flexibility_step(
                pred_action_name=pred_action_name,
                step=step,
                previous_stage_only_actions=previous_stage_only_actions,
                current_stage_actions=current_stage_actions,
            )

            transition_flag = is_transition_step(steps, i)
            step_weight = TRANSITION_STEP_WEIGHT if transition_flag else BASE_STEP_WEIGHT

            weighted_sum += flex_score * step_weight
            weight_total += step_weight

            if transition_flag:
                transition_sum += flex_score
                transition_count += 1

            if pred_action_name != step["correct_action"]:
                wrong_total += 1
                if label == "perseveration_error":
                    perseveration_errors += 1

            row = {
                "step": i + 1,
                "task_id": step["task_id"],
                "stage": current_stage,
                "transition_step": transition_flag,
                "weight": step_weight,
                "pred_action": pred_action_name,
                "correct_action": step["correct_action"],
                "label": label,
                "flex_score": flex_score,
                "running_flex": weighted_sum / weight_total if weight_total > 0 else 0.0,
                "previous_stage_only_actions": sorted(previous_stage_only_actions),
            }
            logs.append(row)

            if VERBOSE:
                print(
                    f"[{i+1:02d}/{total:02d}] "
                    f"stage={current_stage:<6} "
                    f"transition={int(transition_flag)} "
                    f"score={flex_score:.2f} "
                    f"label={label:<22} "
                    f"run={row['running_flex']:.3f}"
                )
                print(f"  pred   : {pred_action_name}")
                print(f"  correct: {step['correct_action']}")
                print(f"  prev-stage-only actions: {sorted(previous_stage_only_actions)}")
                print()

    flexibility_score = weighted_sum / weight_total if weight_total > 0 else 0.0
    transition_flex_score = transition_sum / transition_count if transition_count > 0 else 0.0
    perseveration_rate = perseveration_errors / wrong_total if wrong_total > 0 else 0.0

    overall_flexibility = (
        W_FLEX * flexibility_score +
        W_TRANSITION * transition_flex_score +
        W_PERSEV * (1.0 - perseveration_rate)
    )

    print("\n=== FINAL FLEXIBILITY METRICS ===")
    print(f"{'flexibility_score':24s}: {flexibility_score:.3f}")
    print(f"{'transition_flex_score':24s}: {transition_flex_score:.3f}")
    print(f"{'perseveration_rate':24s}: {perseveration_rate:.3f}")
    print(f"{'overall_flexibility':24s}: {overall_flexibility:.3f}")

    metrics = {
        "flexibility_score": flexibility_score,
        "transition_flex_score": transition_flex_score,
        "perseveration_rate": perseveration_rate,
        "overall_flexibility": overall_flexibility,
    }

    return metrics, logs


# =========================================================
# TASK
# =========================================================
@kbench.task(name="europa_flexibility_stage_transition_weighted")
def europa_flexibility(llm, steps) -> float:
    """
    Main leaderboard score: weighted overall flexibility score.
    """
    try:
        metrics, logs = run_flexibility_episode(llm, steps)
        return float(metrics["overall_flexibility"])
    except Exception as e:
        print(f"Task-level error: {e}")
        return 0.0


# =========================================================
# RUN
# =========================================================
validate_action_coverage(steps)

df = pd.DataFrame([{"steps": steps}])

print("Loaded total scenarios:", len(steps))
print("BASE_STEP_WEIGHT:", BASE_STEP_WEIGHT)
print("TRANSITION_STEP_WEIGHT:", TRANSITION_STEP_WEIGHT)
print("USE_ACTION_IDS:", USE_ACTION_IDS)
print("Overall weights:", {
    "flexibility_score": W_FLEX,
    "transition_flex_score": W_TRANSITION,
    "1-perseveration_rate": W_PERSEV,
})

results = europa_flexibility.evaluate(
    llm=[kbench.llm],
    evaluation_data=df,
    n_jobs=1,
)

display(results.as_dataframe())

=== ACTION COVERAGE CHECK ===
Missing correct_action: []
Missing valid_actions : []
Loaded total scenarios: 21
BASE_STEP_WEIGHT: 1.0
TRANSITION_STEP_WEIGHT: 2.5
USE_ACTION_IDS: True
Overall weights: {'flexibility_score': 0.35, 'transition_flex_score': 0.45, '1-perseveration_rate': 0.2}
[01/21] stage=stage1 transition=0 score=1.00 label=exact_correct          run=1.000
  pred   : launch_from_earth
  correct: launch_from_earth
  prev-stage-only actions: []

[02/21] stage=stage1 transition=0 score=1.00 label=exact_correct          run=1.000
  pred   : continue_burn
  correct: continue_burn
  prev-stage-only actions: []

[03/21] stage=stage2 transition=1 score=1.00 label=exact_correct          run=1.000
  pred   : separate_stage1
  correct: separate_stage1
  prev-stage-only actions: ['continue_burn', 'launch_from_earth']

[04/21] stage=stage2 transition=0 score=1.00 label=exact_correct          run=1.000
  pred   : coast_transfer
  correct: coast_transfer
  prev-stage-only actions: ['conti

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:  1.1min finished


,llm,steps,result,id
run_id,,,,
Run #1,🤖 google/gemini-3-flash-preview,"[{'task_id': 'stage1_01_launch_from_earth', 's...",0.993,0
